# Lab 48 — PEPS y Hubbard 2D en Cilindro

Este laboratorio implementa dos herramientas complementarias para la simulación cuántica de sistemas 2D:

1. **PEPS** (Projected Entangled Pair States) — el análogo 2D del MPS
2. **Modelo de Hubbard en cilindro** — mapeado a 1D para ED y DMRG

**Objetivos:**
- Construir tensores PEPS y contraerlos exactamente para sistemas pequeños
- Implementar el Hamiltoniano de Hubbard 2D via second quantization
- Calcular el gap de carga (transición metal-aislante de Mott)
- Estudiar la doble ocupación y el factor de estructura de espín en función de U/t

**Prereqs:** Lab 46 (MPS/TEBD), Lab 47 (iDMRG), Lab 44 (Hubbard VQE 1D)

## 0. Imports y Configuración

Este laboratorio usa ED (diagonalización exacta) con NumPy y tensores para simular redes PEPS y el modelo de Hubbard 2D. No requiere Qiskit — ilustra el puente entre métodos clásicos de muchos cuerpos y la simulación cuántica.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.sparse import csr_matrix, kron as skron, eye as seye
from scipy.sparse.linalg import eigsh
from scipy.linalg import eigh
import warnings
warnings.filterwarnings('ignore')

print('Entorno listo para PEPS y Hubbard 2D')
print(f'  numpy {np.__version__}')

## 1. PEPS — Projected Entangled Pair States

Un MPS en 1D tiene tensores con 3 índices: `[físico, bono-izq, bono-der]`.  
Un **PEPS** en 2D tiene tensores con 5 índices: `[físico, bono-izq, bono-der, bono-arriba, bono-abajo]`.

```
  ┌───────────────────────────────────┐
  │   Red PEPS en retícula Lx × Ly   │
  │                                   │
  │  [A00]──α──[A01]──α──[A02]        │
  │    │         │         │           │
  │    γ         δ         ε           │
  │    │         │         │           │
  │  [A10]──β──[A11]──β──[A12]        │
  │    │         │         │           │
  │   ...       ...       ...          │
  └───────────────────────────────────┘
```

- Cada tensor `A[i,j]` tiene dimensión de bono **D** → parámetro de precisión
- Estado completo: `4^N` amplitudes vs `N·d·D^4` parámetros del PEPS
- Para D=2, N=16: 65536 amplitudes → 16·4·16 = 1024 parámetros

**Area law 2D (Hastings 2006):** estados fundamentales de Hamiltonianos locales 2D gapped satisfacen  
`S(A) ≤ const × |∂A|` (entropía proporcional al perímetro, no al área).  
⟹ PEPS con D finito puede representarlos eficientemente, mientras MPS necesita χ ~ 2^W (exponencial en el ancho).

In [ ]:
class PEPS:
    """
    PEPS en retícula 2D Lx × Ly con dimensión de bono D.
    
    Tensor en (i,j): shape (d, D_left, D_right, D_up, D_down)
    Índices de borde tienen dimensión 1.
    """

    def __init__(self, Lx, Ly, d, D, seed=42):
        self.Lx, self.Ly, self.d, self.D = Lx, Ly, d, D
        rng = np.random.default_rng(seed)
        self.tensors = {}
        for i in range(Lx):
            for j in range(Ly):
                Dl = 1 if j == 0      else D
                Dr = 1 if j == Ly - 1 else D
                Du = 1 if i == 0      else D
                Dd = 1 if i == Lx - 1 else D
                self.tensors[(i, j)] = rng.standard_normal((d, Dl, Dr, Du, Dd))

    @classmethod
    def product_state(cls, Lx, Ly, local_state):
        """PEPS de estado producto: D=1, tensor local = amplitudes del estado local."""
        d = len(local_state)
        peps = cls(Lx, Ly, d, 1)
        for i in range(Lx):
            for j in range(Ly):
                T = np.zeros_like(peps.tensors[(i, j)])
                for s, amp in enumerate(local_state):
                    T[s, 0, 0, 0, 0] = amp
                peps.tensors[(i, j)] = T
        return peps

    def to_statevector_2x2(self):
        """
        Contracción exacta para Lx=Ly=2.

        Índices de bono en la red 2×2:
          α: (0,0)──(0,1)   β: (1,0)──(1,1)
          γ: (0,0)──(1,0)   δ: (0,1)──(1,1)

        Contracción: state[s00,s01,s10,s11] = Σ_{α,β,γ,δ} T00[s,α,γ] T01[s,α,δ] T10[s,β,γ] T11[s,β,δ]
        """
        assert self.Lx == 2 and self.Ly == 2, "Solo válido para 2×2"
        d, D = self.d, self.D

        # Extraer tensores quitando índices triviales (dimensión 1)
        # A[(0,0)]: (d, 1, D, 1, D) → (d, D_right=α, D_down=γ)
        T00 = self.tensors[(0,0)].reshape(d, D, D)
        # A[(0,1)]: (d, D, 1, 1, D) → (d, D_left=α, D_down=δ)
        T01 = self.tensors[(0,1)].reshape(d, D, D)
        # A[(1,0)]: (d, 1, D, D, 1) → (d, D_right=β, D_up=γ)
        T10 = self.tensors[(1,0)].reshape(d, D, D)
        # A[(1,1)]: (d, D, 1, D, 1) → (d, D_left=β, D_up=δ)
        T11 = self.tensors[(1,1)].reshape(d, D, D)

        # Σ_{α,γ,β,δ} T00[i,α,γ] T01[j,α,δ] T10[k,β,γ] T11[l,β,δ]
        state = np.einsum('iag,jad,kbg,lbd->ijkl', T00, T01, T10, T11)
        state = state.reshape(d**4)
        norm = np.linalg.norm(state)
        return state / norm if norm > 1e-14 else state

    def entanglement_entropy_2x2_horizontal(self):
        """Entropía de entrelazamiento del corte horizontal (fila 0 vs fila 1) para 2×2."""
        psi = self.to_statevector_2x2()
        M = psi.reshape(self.d**2, self.d**2)
        _, S, _ = np.linalg.svd(M, full_matrices=False)
        lam2 = S**2
        lam2 = lam2[lam2 > 1e-14]
        return -np.sum(lam2 * np.log2(lam2))


# Ejemplo: estado producto |↑⟩⊗4 en red 2×2
peps_prod = PEPS.product_state(2, 2, local_state=[1.0, 0.0])  # d=2, todos en |0⟩
psi_prod = peps_prod.to_statevector_2x2()
print('Estado producto |0000⟩:')
print(f'  Amplitud |0000⟩ = {psi_prod[0]:.4f}  (debe ser 1.0)')
print(f'  Norma = {np.linalg.norm(psi_prod):.6f}')
print(f'  Entropía de entrelazamiento = {peps_prod.entanglement_entropy_2x2_horizontal():.6f} ebits  (debe ser 0)')

# PEPS con D=2 aleatorio: entropía no nula
peps_ent = PEPS(2, 2, d=2, D=2)
S_ent = peps_ent.entanglement_entropy_2x2_horizontal()
print(f'\nPEPS aleatorio (D=2): entropía = {S_ent:.4f} ebits')

### Escalado de Entropía con el Rango de Vínculo $D$

El rango de vínculo $D$ determina la capacidad del PEPS para representar entrelazamiento. La entropía de entrelazamiento escala como $S \sim \log D$ para estados aleatorios, mientras que los estados físicos de baja energía satisfacen la **ley de área** $S \propto L$ (perímetro del bloque).

In [ ]:
# Escalado de entropía con D para PEPS 2×2 aleatorio
D_vals = [1, 2, 4, 8, 16]
S_vals = []

for D in D_vals:
    peps = PEPS(2, 2, d=2, D=D, seed=0)
    S = peps.entanglement_entropy_2x2_horizontal()
    S_vals.append(S)

# Cota teórica: S ≤ log₂(D²) = 2 log₂(D)
S_bound = [2 * np.log2(D) for D in D_vals]

plt.figure(figsize=(7, 4))
plt.plot(D_vals, S_vals, 'bo-', lw=2, ms=8, label='S (PEPS aleatorio)')
plt.plot(D_vals, S_bound, 'r--', lw=2, label='Cota 2·log₂(D)')
plt.xscale('log', base=2)
plt.xlabel('Dimensión de bono D', fontsize=12)
plt.ylabel('Entropía de entrelazamiento (ebits)', fontsize=12)
plt.title('PEPS 2×2: entropía vs bond dimension', fontsize=13)
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print('Cota area law 2D: S(A) ≤ const × |∂A|')
print(f'Para corte horizontal en 2×2: |∂A| = 2 bonds → S ≤ 2·log₂(D) = {2*np.log2(D_vals[-1]):.2f} ebits')

## 2. Modelo de Hubbard 2D

El Hamiltoniano de Hubbard en retícula cuadrada $L_x \times L_y$:

$$H = -t \sum_{\langle i,j \rangle, \sigma} \left( c^\dagger_{i\sigma} c_{j\sigma} + \text{h.c.} \right) + U \sum_i n_{i\uparrow} n_{i\downarrow}$$

- $t > 0$: amplitud de hopping (energía cinética)
- $U > 0$: repulsión de Coulomb on-site (energía potencial)
- $\langle i,j \rangle$: primeros vecinos en la red

### Transformación de Jordan-Wigner

Ordenamos los modos como: $(0\uparrow, 0\downarrow, 1\uparrow, 1\downarrow, \ldots)$  
Cada modo es un qubit: el estado de los $N$ sitios vive en $\{0,1\}^{2N}$.

El operador de creación en el modo $k$ con string de Jordan-Wigner:
$$c^\dagger_k = \left(\prod_{j<k} Z_j\right) \otimes \sigma^+_k$$

El signo JW cuenta los fermiones ocupados antes del modo $k$:
$$c^\dagger_k |\mathbf{n}\rangle = (-1)^{n_0 + n_1 + \cdots + n_{k-1}} |\mathbf{n}\text{ con bit }k\text{ encendido}\rangle$$

### Transición metal-aislante de Mott

A semillenado ($N_e = N_{\text{sitios}}$):
- $U/t \ll 1$: metal de Fermi (electrones itinerantes)
- $U/t \gg 1$: aislante de Mott (electrones localizados, un electrón por sitio)

En 1D: la transición es en $U_c = 0$ (toda $U > 0$ es aislante — teorema de Lieb-Wu).  
En 2D cuadrada: $U_c/t \approx 3.8$ (Monte Carlo cuántico, LeBlanc et al. 2015).

**Gap de carga:**
$$\Delta_c = E_0(N_e+1) + E_0(N_e-1) - 2E_0(N_e)$$
Es cero en el metal y positivo en el aislante.

### Operadores de Creación/Destrucción en Segunda Cuantización

Los operadores fermiónicos se construyen usando la **transformación Jordan-Wigner** que mapea operadores de creación/destrucción a cadenas de matrices de Pauli, preservando las relaciones de anticonmutación $\{c_i, c_j^\dagger\} = \delta_{ij}$.

In [ ]:
def build_cdag(k, n_modes):
    """
    Operador de creación c†_k en el espacio de 2^n_modes estados.
    Implementación directa por manipulación de bits (O(2^n_modes)).
    """
    dim = 2**n_modes
    rows, cols, vals = [], [], []
    for state in range(dim):
        if not (state >> k) & 1:   # modo k desocupado
            new_state = state | (1 << k)
            # Sign JW: número de modos ocupados por debajo de k
            n_below = bin(state & ((1 << k) - 1)).count('1')
            sign = (-1)**n_below
            rows.append(new_state)
            cols.append(state)
            vals.append(float(sign))
    return csr_matrix((vals, (rows, cols)), shape=(dim, dim), dtype=float)


def build_number_op(k, n_modes):
    """Operador número n_k = c†_k c_k."""
    dim = 2**n_modes
    diag = np.array([(state >> k) & 1 for state in range(dim)], dtype=float)
    return csr_matrix(np.diag(diag))


def build_hubbard_2d(Lx, Ly, t=1.0, U=4.0):
    """
    Hamiltoniano de Hubbard en retícula Lx×Ly con condiciones de contorno abiertas.

    Ordenamiento de sitios: snake (fila por fila).
    Modos: sitio s spin-up → modo 2s, sitio s spin-down → modo 2s+1.
    """
    N = Lx * Ly
    n_modes = 2 * N
    dim = 2**n_modes

    def site(i, j): return i * Ly + j

    # Bonds en la red 2D
    bonds = []
    for i in range(Lx):
        for j in range(Ly):
            s = site(i, j)
            if j < Ly - 1: bonds.append((s, site(i, j + 1)))    # horizontal
            if i < Lx - 1: bonds.append((s, site(i + 1, j)))    # vertical

    H = csr_matrix((dim, dim), dtype=float)

    # Término de hopping: -t (c†_{j,σ} c_{i,σ} + h.c.)
    for (si, sj) in bonds:
        for sigma in range(2):   # 0=up, 1=down
            ki = 2 * si + sigma
            kj = 2 * sj + sigma
            cd_i = build_cdag(ki, n_modes)
            cd_j = build_cdag(kj, n_modes)
            c_i  = cd_i.T
            c_j  = cd_j.T
            H -= t * (cd_j @ c_i + cd_i @ c_j)

    # Término de Hubbard: U n_{s,↑} n_{s,↓}
    for s in range(N):
        n_up   = build_number_op(2 * s,     n_modes)
        n_down = build_number_op(2 * s + 1, n_modes)
        H += U * (n_up @ n_down)

    return H, n_modes, dim


def build_Ntotal(n_modes):
    """Operador número total N_total = Σ_k n_k."""
    dim = 2**n_modes
    diag = np.array([bin(state).count('1') for state in range(dim)], dtype=float)
    return csr_matrix(np.diag(diag))


print('Construcción del Hamiltoniano de Hubbard 2×2 (U/t=4)...')
H22, n_modes22, dim22 = build_hubbard_2d(2, 2, t=1.0, U=4.0)
print(f'  Sitios: 4 (2×2),  modos: {n_modes22},  dim Hilbert: {dim22}')
print(f'  Elementos no nulos en H: {H22.nnz}')
print(f'  Es simétrica: {(H22 - H22.T).nnz == 0}')

In [ ]:
def ground_energy_sector(H, Ntotal_op, Ne, lam=200.0):
    """
    Energía del estado fundamental con exactamente Ne partículas.
    Usa penalización: H' = H + λ(N_total - Ne)²
    """
    Ne_op = Ne * seye(H.shape[0], format='csr')
    penalty = lam * (Ntotal_op - Ne_op) @ (Ntotal_op - Ne_op)
    H_pen = H + penalty
    evals, _ = eigsh(H_pen, k=1, which='SA')
    # Restar la contribución de la penalización en el sector correcto (= 0)
    return float(evals[0])


def charge_gap(H, Ntotal_op, Ne_half):
    """Gap de carga: Δc = E₀(Ne+1) + E₀(Ne-1) - 2·E₀(Ne)."""
    E0   = ground_energy_sector(H, Ntotal_op, Ne_half)
    Ep1  = ground_energy_sector(H, Ntotal_op, Ne_half + 1)
    Em1  = ground_energy_sector(H, Ntotal_op, Ne_half - 1)
    return Ep1 + Em1 - 2 * E0, E0


# ED para 2×2 barriendo U/t
Lx, Ly = 2, 2
N_sites = Lx * Ly   # 4 sitios
Ne_half = N_sites   # semillenado: 4 electrones en 4 sitios

U_over_t = np.linspace(0.5, 12, 20)
gaps_22   = []
E0_22     = []

print(f'ED para retícula {Lx}×{Ly} (dim={2**(2*N_sites)}), semillenado Ne={Ne_half}:')
Ntotal22 = build_Ntotal(2 * N_sites)

for U in U_over_t:
    H, _, _ = build_hubbard_2d(Lx, Ly, t=1.0, U=U)
    gap, E0 = charge_gap(H, Ntotal22, Ne_half)
    gaps_22.append(gap)
    E0_22.append(E0 / N_sites)
    print(f'  U/t={U:5.1f}:  E₀/sitio={E0/N_sites:.4f}  Δc={gap:.4f}')

## 3. Diagonalización Exacta del Modelo de Hubbard 2D

Usamos ED en la geometría cilíndrica 2×4 con **ordenamiento en serpiente** (snake ordering) que mapea el retículo 2D a una cadena 1D para la construcción del Hamiltoniano.

In [ ]:
# ED para 2×4 en cilindro (snake ordering)
Lx2, Ly2 = 2, 4
N_sites2 = Lx2 * Ly2   # 8 sitios
Ne_half2 = N_sites2    # semillenado

U_over_t2 = np.linspace(0.5, 12, 12)
gaps_24   = []
E0_24     = []

print(f'ED para cilindro {Lx2}×{Ly2} (dim={2**(2*N_sites2)}), semillenado Ne={Ne_half2}:')
Ntotal24 = build_Ntotal(2 * N_sites2)

for U in U_over_t2:
    H, _, _ = build_hubbard_2d(Lx2, Ly2, t=1.0, U=U)
    gap, E0 = charge_gap(H, Ntotal24, Ne_half2)
    gaps_24.append(gap)
    E0_24.append(E0 / N_sites2)
    print(f'  U/t={U:5.1f}:  E₀/sitio={E0/N_sites2:.4f}  Δc={gap:.4f}')

In [ ]:
# Visualización: gap de carga y energía vs U/t
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Panel 1: gap de carga
axes[0].plot(U_over_t,  gaps_22, 'b-o', lw=2, ms=7, label='2×2 (ED exacta)')
axes[0].plot(U_over_t2, gaps_24, 'g-s', lw=2, ms=7, label='2×4 cilindro (ED)')
axes[0].axvline(x=3.8, color='red', ls='--', lw=1.5, label='Uc/t ≈ 3.8 (2D QMC)')
axes[0].set_xlabel('U/t', fontsize=12)
axes[0].set_ylabel('Gap de carga Δc (unidades de t)', fontsize=12)
axes[0].set_title('Transición metal-aislante de Mott', fontsize=13)
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

# Panel 2: energía por sitio
axes[1].plot(U_over_t,  E0_22, 'b-o', lw=2, ms=7, label='2×2')
axes[1].plot(U_over_t2, E0_24, 'g-s', lw=2, ms=7, label='2×4 cilindro')
# Límites: U→0 (banda libre) y U→∞ (aislante de Heisenberg)
axes[1].axhline(y=-4/np.pi, color='purple', ls=':', lw=1.5, label='1D libre: -4t/π per bond')
axes[1].set_xlabel('U/t', fontsize=12)
axes[1].set_ylabel('E₀/sitio (unidades de t)', fontsize=12)
axes[1].set_title('Energía del estado fundamental', fontsize=13)
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)

plt.suptitle('Modelo de Hubbard 2D — Exact Diagonalization', fontsize=14)
plt.tight_layout()
plt.show()

## 4. Doble Ocupación y Factor de Estructura

La **doble ocupación** $D_\text{occ} = \langle n_{i\uparrow} n_{i\downarrow}\rangle$ y el **factor de estructura de espín** $S(k)$ son observables clave del modelo de Hubbard. En el límite de acoplamiento fuerte ($U/t \gg 1$), el sistema se convierte en un antiferromagneto.

In [ ]:
def double_occupancy_and_structure_factor(Lx, Ly, U_vals, t=1.0):
    """
    Calcula doble ocupación ⟨D⟩ y factor de estructura AF S(π,π)
    como función de U/t para semillenado.
    """
    N = Lx * Ly
    Ne = N
    n_modes = 2 * N
    Ntotal = build_Ntotal(n_modes)

    def site(i, j): return i * Ly + j

    D_avg_list = []
    Sq_list    = []

    for U in U_vals:
        H, _, dim = build_hubbard_2d(Lx, Ly, t=t, U=U)
        Ne_op  = Ne * seye(dim, format='csr')
        pen    = 200.0 * (Ntotal - Ne_op) @ (Ntotal - Ne_op)
        evals, evecs = eigsh(H + pen, k=1, which='SA')
        psi = evecs[:, 0]

        # Doble ocupación media: D = (1/N) Σ_s ⟨n_{s,↑} n_{s,↓}⟩
        D_avg = 0.0
        for s in range(N):
            n_up   = build_number_op(2 * s,     n_modes)
            n_down = build_number_op(2 * s + 1, n_modes)
            D_avg += float(psi @ (n_up @ n_down) @ psi)
        D_avg /= N
        D_avg_list.append(D_avg)

        # Factor de estructura antiferromagnético S(π,π)
        # S^z_{i} = (n_{i,↑} - n_{i,↓}) / 2
        Sq = 0.0
        for i in range(Lx):
            for j in range(Ly):
                s1 = site(i, j)
                phase1 = (-1)**(i + j)
                Sz1 = 0.5 * (build_number_op(2*s1, n_modes) -
                             build_number_op(2*s1+1, n_modes))
                for ii in range(Lx):
                    for jj in range(Ly):
                        s2 = site(ii, jj)
                        phase2 = (-1)**(ii + jj)
                        Sz2 = 0.5 * (build_number_op(2*s2, n_modes) -
                                     build_number_op(2*s2+1, n_modes))
                        Sq += phase1 * phase2 * float(psi @ (Sz1 @ Sz2) @ psi)
        Sq /= N
        Sq_list.append(Sq)

    return D_avg_list, Sq_list


print('Calculando doble ocupación y S(π,π) para 2×2...')
U_fine = np.linspace(0.5, 10, 10)
D_22, Sq_22 = double_occupancy_and_structure_factor(2, 2, U_fine)

In [ ]:
# Visualización: doble ocupación y factor de estructura
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Panel 1: doble ocupación
axes[0].plot(U_fine, D_22, 'b-o', lw=2, ms=8, label='2×2 (ED)')
axes[0].axhline(y=0.25, color='gray', ls=':', lw=1.5, label='Límite U→0: D=0.25')
axes[0].axhline(y=0.0,  color='red',  ls=':', lw=1.5, label='Límite U→∞: D→0')
axes[0].set_xlabel('U/t', fontsize=12)
axes[0].set_ylabel('Doble ocupación ⟨D⟩', fontsize=12)
axes[0].set_title('Localización de Mott', fontsize=13)
axes[0].legend(fontsize=10)
axes[0].set_ylim(-0.02, 0.30)
axes[0].grid(True, alpha=0.3)

# Panel 2: factor de estructura antiferromagnético
axes[1].plot(U_fine, Sq_22, 'g-s', lw=2, ms=8, label='2×2 (ED)')
axes[1].axvline(x=3.8, color='red', ls='--', lw=1.5, label='Uc/t ≈ 3.8 (2D)')
axes[1].set_xlabel('U/t', fontsize=12)
axes[1].set_ylabel('S(π,π) / sitio', fontsize=12)
axes[1].set_title('Orden antiferromagnético', fontsize=13)
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)

plt.suptitle('Transición de Mott: diagnósticos locales', fontsize=14)
plt.tight_layout()
plt.show()

print('Resumen de la transición de Mott:')
print('  U/t    D_occ    S(π,π)')
for U, D, S in zip(U_fine, D_22, Sq_22):
    print(f'  {U:5.1f}  {D:.4f}   {S:.4f}')

## 5. Diagrama de Fase Metal-Aislante

En 2D, el modelo de Hubbard exhibe una **transición metal-aislante de Mott** al aumentar $U/t$. Para medio llenado ($n=1$), la transición ocurre a un valor crítico $U_c/t \approx 5$-$8$ (dependiente de la geometría).

In [ ]:
# Diagrama de fase cualitativo: metal-aislante en función de U/t y llenado
fig, ax = plt.subplots(figsize=(9, 5))

U_plot = np.linspace(0, 14, 200)

# Gap de carga interpolado (2×2)
from scipy.interpolate import interp1d
f_gap = interp1d(U_over_t, np.clip(gaps_22, 0, None), kind='cubic', fill_value='extrapolate')
gap_smooth = np.clip(f_gap(U_plot), 0, None)

ax.fill_between(U_plot, 0, gap_smooth, alpha=0.3, color='red', label='Aislante de Mott (Δc > 0)')
ax.fill_between(U_plot, gap_smooth, 0, where=(gap_smooth <= 0.05), alpha=0.3,
                color='blue', label='Metal (Δc ≈ 0)')
ax.plot(U_plot, gap_smooth, 'k-', lw=2, label='Gap de carga Δc(U/t)')
ax.axvline(x=3.8, color='red', ls='--', lw=2, label='Uc/t = 3.8 (2D, QMC)')
ax.axvline(x=0.0, color='blue', ls='--', lw=2, label='Uc/t = 0 (1D, Lieb-Wu)')

ax.set_xlabel('U/t (interacción / hopping)', fontsize=12)
ax.set_ylabel('Gap de carga Δc (en unidades de t)', fontsize=12)
ax.set_title('Diagrama de fase — Hubbard 2D a semillenado (2×2 ED)', fontsize=13)
ax.legend(fontsize=9, loc='upper left')
ax.set_xlim(0, 14)
ax.set_ylim(-0.1, None)
ax.grid(True, alpha=0.3)

# Anotaciones
ax.annotate('Metal de Fermi\n(electrones itinerantes)',
            xy=(1.5, 0.05), fontsize=10, color='blue',
            ha='center')
ax.annotate('Aislante de Mott\n(un e⁻ por sitio)',
            xy=(10, 1.0), fontsize=10, color='darkred',
            ha='center')

plt.tight_layout()
plt.show()

## 6. Comparativa PEPS vs MPS

Para estados 2D, los MPS requieren un rango de vínculo exponencialmente grande en la anchura del cilindro ($D \sim e^{L_y}$), mientras que los PEPS escalan polinómicamente. Esta diferencia hace que PEPS sean el ansatz natural para sistemas 2D.

In [ ]:
# Tabla resumen: PEPS vs MPS para representar estados 2D
print('=' * 65)
print('RESUMEN: PEPS vs MPS para sistemas 2D')
print('=' * 65)
print(f'{"Propiedad":<35} {"MPS":<15} {"PEPS"}')
print('-' * 65)
rows_summary = [
    ('Dimensión de bono',           'χ',        'D'),
    ('Nº parámetros',               'O(Nχ²)',   'O(N·d·D⁴)'),
    ('Contracción',                 'O(Nχ³)',   '#P-difícil (exacta)'),
    ('Area law 1D (gapped)',        'χ=O(1)',   '—'),
    ('Area law 2D (gapped)',        'χ~2^W',    'D=O(1)'),
    ('Estados críticos 1D',         'χ~L^c',    '—'),
    ('Superconductividad (2D)',      'χ→∞',      'D finito aprox.'),
    ('Implementación práctica',     'Exacta',   'Boundary MPS aprox.'),
]
for name, mps, peps in rows_summary:
    print(f'{name:<35} {mps:<15} {peps}')

print()
print('RESUMEN: Transición de Mott en Hubbard 2D')
print('-' * 65)
print(f'  2×2 ED: Δc(U=4) = {float(f_gap(4)):.4f} t')
print(f'  2×4 ED: Δc(U=4) = {gaps_24[np.argmin(np.abs(U_over_t2 - 4))]:.4f} t')
print(f'  Referencia 2D (QMC): Uc/t ≈ 3.8')
print(f'  Límite 1D (Lieb-Wu): Uc/t = 0 (todo U>0 es aislante)')
print()
print('Coste computacional:')
print(f'  2×2: dim=256,   t_build ≪ 1s')
print(f'  2×4: dim=65536, t_build ≈ segundos')
print(f'  2×6: dim=2^24≈16M → necesita DMRG (χ~100-200)')